# Skill Dimension Loader

This notebook maintains the `warehouse.dim_skill` dimension table.

**Purpose**: Track skills from the semantic skill catalog

**Key Features**:
* Stable surrogate keys for skills
* Skill hierarchies and categories
* Skill metadata (proficiency levels, demand metrics)

**Source**: `semantic.sem_skill_catalog`

**Target**: `workspace.warehouse.dim_skill`

In [0]:
# Load skill taxonomy directly from governed metadata
# This ensures warehouse dimension reflects the canonical skill taxonomy

from pyspark.sql import functions as F

METADATA_BASE = "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata"

# Load skill taxonomy
skills_df = spark.read.csv(
    f"{METADATA_BASE}/canonical_skills.csv",
    header=True,
    inferSchema=True
)

# Transform for warehouse dimension
skill_extract_df = skills_df.select(
    F.col("skill_key").alias("skill_id"),
    F.col("canonical_skill").alias("skill_name"),
    F.col("skill_category"),
    F.coalesce(F.col("sector_key"), F.lit("CROSS_SECTOR")).alias("sector_key"),
    F.lit(1).alias("skill_level"),
    F.lit(None).cast("string").alias("parent_skill_id"),
    F.when(F.col("skill_category") == "Technical", True).otherwise(False).alias("is_technical"),
    F.lit(0.0).alias("demand_score"),
    F.lit(True).alias("is_active"),
    F.split(F.col("aliases"), "\\|").alias("aliases")
)

skill_extract_df.createOrReplaceTempView("skill_extract")

print(f"Loaded {skill_extract_df.count()} skills from metadata taxonomy")
display(skill_extract_df)

In [0]:
%sql
-- Generate or maintain stable surrogate keys
CREATE OR REPLACE TEMP VIEW skill_with_keys AS
SELECT
  COALESCE(d.skill_sk, ROW_NUMBER() OVER (ORDER BY s.skill_id) + COALESCE(max_sk, 0)) as skill_sk,
  s.skill_id,
  s.skill_name,
  s.skill_category,
  s.sector_key,
  s.skill_level,
  s.parent_skill_id,
  s.aliases,
  COALESCE(s.is_technical, FALSE) as is_technical,
  COALESCE(s.demand_score, 0.0) as demand_score,
  COALESCE(s.is_active, TRUE) as is_active,
  CURRENT_TIMESTAMP() as updated_at
FROM skill_extract s
LEFT JOIN workspace.warehouse.dim_skill d
  ON s.skill_id = d.skill_id
CROSS JOIN (
  SELECT COALESCE(MAX(skill_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_skill
) max_keys;

In [0]:
%sql
-- Merge into target dimension (SCD Type 1)
-- First, ensure table exists with updated schema
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_skill (
  skill_sk BIGINT,
  skill_id STRING,
  skill_name STRING,
  skill_category STRING,
  sector_key STRING,
  skill_level INT,
  parent_skill_id STRING,
  aliases ARRAY<STRING>,
  is_technical BOOLEAN,
  demand_score DOUBLE,
  is_active BOOLEAN,
  updated_at TIMESTAMP
) USING DELTA;

MERGE INTO workspace.warehouse.dim_skill target
USING skill_with_keys source
ON target.skill_id = source.skill_id
WHEN MATCHED THEN UPDATE SET
  target.skill_name = source.skill_name,
  target.skill_category = source.skill_category,
  target.sector_key = source.sector_key,
  target.skill_level = source.skill_level,
  target.parent_skill_id = source.parent_skill_id,
  target.aliases = source.aliases,
  target.is_technical = source.is_technical,
  target.demand_score = source.demand_score,
  target.is_active = source.is_active,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  skill_sk,
  skill_id,
  skill_name,
  skill_category,
  sector_key,
  skill_level,
  parent_skill_id,
  aliases,
  is_technical,
  demand_score,
  is_active,
  updated_at
) VALUES (
  source.skill_sk,
  source.skill_id,
  source.skill_name,
  source.skill_category,
  source.sector_key,
  source.skill_level,
  source.parent_skill_id,
  source.aliases,
  source.is_technical,
  source.demand_score,
  source.is_active,
  source.updated_at
);

In [0]:
%sql
-- Validate skill dimension
SELECT 
  COUNT(*) as total_skills,
  COUNT(DISTINCT skill_category) as skill_categories,
  SUM(CASE WHEN is_technical THEN 1 ELSE 0 END) as technical_skills,
  AVG(demand_score) as avg_demand_score,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_skills
FROM workspace.warehouse.dim_skill;

-- Sample skills
SELECT 
  skill_sk,
  skill_name,
  skill_category,
  is_technical,
  demand_score,
  is_active
FROM workspace.warehouse.dim_skill
ORDER BY demand_score DESC
LIMIT 20;